# final 综合题——时序预测全流程（load_curve.csv）

> 数据：`load_curve.csv`，2026 年 1~3 月、3 个台区、小时级负荷（6480 行，
> 负荷值 48 个缺口）。任务：**预测 STATION_A_01 的小时负荷**。
>
> 对应竞赛评分点全景：清洗（pandas 跨章）→ 特征工程（ch02）→ 切分（ch01）
> → 多模型（ch04）→ 调参（ch09）→ 误差分析（ch08）→ 可视化（ch11）
> → 落盘验收（ch13）。**每一步都有 assert 兜底，跑通即满分骨架**。

**流程总览**

| 步骤 | 内容 | 关键决策 |
|---|---|---|
| ① 清洗 | datetime / 排序 / 组内插值 | 不跨台区借数 |
| ② 特征 | hour / dow / lag24 / lag168 / roll24_mean | 滚动窗口先 shift(1) 防自泄漏 |
| ③ 切分 | 前 80% 训练 / 后 20% 测试 | 基线 = lag24（MAE 48.15） |
| ④ 对比 | LR / Ridge / RF / GBR | 树集成全面胜出 |
| ⑤ 调参 | TimeSeriesSplit(3) + 网格 | 折不许 shuffle |
| ⑥ 解剖 | 重要性 + 分时段误差 | 用数据说话 |
| ⑦ 交付 | joblib 落盘 + 指纹验收 | 只 load 不 fit |

**难点索引**

| 难点 | 为什么是坑 | 位置 |
|---|---|---|
| 滚动窗口自泄漏 | roll24_mean 不 shift(1) 就把答案喂给模型 | §F.2 |
| 用 train_test_split 切时序 | 未来数据倒灌训练集 | §F.3 |
| cv 分数比 test 差 | TimeSeriesSplit 早期折又旧又短，是常态 | §F.5 |
| 基线被忽视 | lag24 MAE 48.15 是模型必须打败的及格线 | §F.3 |

In [ ]:
from pathlib import Path

import hashlib
import os
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB",
                                   "Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("data")

In [ ]:
# 台区选定与指标工具（写在 @@todo 之外）
FEATS = ["hour", "dow", "温度", "是否节假日", "lag24", "lag168", "roll24_mean"]
TARGET = "负荷值"
STATION = "STATION_A_01"


def mae_rmse_r2(yt, yp):
    """回归三件套指标。"""
    return (round(float(mean_absolute_error(yt, yp)), 4),
            round(float(np.sqrt(mean_squared_error(yt, yp))), 4),
            round(float(r2_score(yt, yp)), 4))


def pred_md5(y_pred):
    """预测指纹：落盘验收用。"""
    return hashlib.md5(np.asarray(y_pred).tobytes()).hexdigest()


print("sklearn", sklearn.__version__, "| 目标台区:", STATION)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def one_line(tag, yt, yp):
    """一行报 MAE / RMSE / R2，返回三元组。"""
    mae, rmse, r2 = mae_rmse_r2(yt, yp)
    print(f"{tag}: MAE={mae} RMSE={rmse} R2={r2}")
    return mae, rmse, r2


print("脚手架就绪：one_line / mae_rmse_r2 / pred_md5")

## F.1 清洗：先排序再插值，组内不跨界

48 个缺口全部来自负荷值。`groupby("台区编号")` 后组内插值，
**不允许跨台区借数**；`limit_direction="both"` 保证首尾缺口也能补。

In [ ]:
# 清洗：48 个缺口全部来自负荷值，组内插值补齐
raw = pd.read_csv(DATA / "load_curve.csv")
n_missing_before = int(raw["负荷值"].isna().sum())
raw["时间戳"] = pd.to_datetime(raw["时间戳"])
raw = raw.sort_values(["台区编号", "时间戳"]).reset_index(drop=True)
raw["负荷值"] = raw.groupby("台区编号")["负荷值"].transform(lambda s: s.interpolate(limit_direction="both"))
print("缺失:", n_missing_before, "→", int(raw["负荷值"].isna().sum()))
print("解读: 时序插值必须先排序再插；groupby 内插保证不跨台区借数——"
      "『先清洗后建模』这 1 步在综合题里单独占分")

assert n_missing_before == 48 and int(raw["负荷值"].isna().sum()) == 0

## F.2 特征工程：给模型时序的眼睛

lag24（昨天同时刻）/ lag168（上周同一时刻）/ 前 24 小时滚动均值。
**滚动均值必须先 `shift(1)`**——把当前时刻从窗口里抠掉，否则就是
自我泄漏；`dropna` 甩掉 lag168 造成的 168 行 warm-up。

In [ ]:
# 特征工程：让模型看见「昨天同时刻」和「上周同一时刻」
g = raw[raw["台区编号"] == STATION].copy().reset_index(drop=True)
g["hour"] = g["时间戳"].dt.hour
g["dow"] = g["时间戳"].dt.dayofweek
g["lag24"] = g["负荷值"].shift(24)
g["lag168"] = g["负荷值"].shift(168)
g["roll24_mean"] = g["负荷值"].shift(1).rolling(24).mean()
g = g.dropna().reset_index(drop=True)
print("特征工程后:", g.shape, "| 特征:", FEATS)
print("解读: 滚动均值必须先 shift(1)——把当前时刻从窗口里抠掉，否则就是自我泄漏；"
      "dropna 甩掉 168 行 warm-up")

assert g.shape == (1992, 10)

## F.3 切分与基线：时序没有 train_test_split 的份

前 80% 训练（1593）、后 20% 测试（399）。训练段均值 651 vs 测试段
743——**分布漂移是时序任务常态**。基线「昨天同时刻」MAE 48.1487：
模型必须先打败它才有存在意义。

In [ ]:
# 时序切分：时间序列没有 train_test_split 的份，基线先行
cut = int(len(g) * 0.8)
tr, te = g.iloc[:cut], g.iloc[cut:]
Xtr, ytr = tr[FEATS], tr[TARGET]
Xte, yte = te[FEATS], te[TARGET]
base_mae = one_line("基线 lag24", yte, te["lag24"])[0]
print("训练段均值:", round(float(ytr.mean()), 4), "| 测试段均值:", round(float(yte.mean()), 4))
print("解读: 前 80% 训练后 20% 测试，训练均值 651 vs 测试均值 743——分布漂移下"
      "基线 MAE 48.15 是『模型必须打败的及格线』")

assert cut == 1593 and base_mae == 48.1487

## F.4 多模型对比

LR 34.22 / Ridge 34.22 / RF100 31.85 / GBR100 30.93——树集成全面胜出，
GBR 领跑。所有模型都跑赢基线，对比才有效。

In [ ]:
# 多模型对比：一张表见高下
models = {
    "LR": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RF100": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GBR100": GradientBoostingRegressor(n_estimators=100, random_state=42),
}
results = {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    results[name] = one_line(name, yte, model.predict(Xte))
print("结论: 树集成 > 线性 > 基线；GBR 以 MAE 30.93 领跑——"
      "时序平滑信号上残差接力更顺")

assert min(v[0] for v in results.values()) < base_mae

## F.5 难点深挖：时序调参的 CV 纪律

`TimeSeriesSplit(3)`：折只增不重排。RF best {max_depth: 10,
n_estimators: 200}，cv MAE 48.37 比 test 31.13 惨得多——**早期折又旧
又短是常态，不是调参失败**。GBR 网格后 test MAE 30.0752 / R² 0.9722，
全场冠军；相对基线误差降 37.5%。

In [ ]:
# RF 调参：TimeSeriesSplit 折不许 shuffle
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [6, 10, None]},
    cv=TimeSeriesSplit(n_splits=3), scoring="neg_mean_absolute_error", n_jobs=-1)
gs_rf.fit(Xtr, ytr)
print("best:", gs_rf.best_params_, "| cv MAE:", round(float(-gs_rf.best_score_), 4))
print("解读: cv MAE 48.37 比 test 31.13 惨得多——TimeSeriesSplit 的早期折又旧又短，"
      "分数天然更难看。这是时序 CV 的常态，不是调参失败")

assert gs_rf.best_params_ == {"max_depth": 10, "n_estimators": 200}

In [ ]:
# GBR 调参：冠军加冕
gs_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {"learning_rate": [0.05, 0.1], "n_estimators": [100, 200]},
    cv=TimeSeriesSplit(n_splits=3), scoring="neg_mean_absolute_error", n_jobs=-1)
gs_gb.fit(Xtr, ytr)
gb_test = one_line("GBR 调参后", yte, gs_gb.best_estimator_.predict(Xte))
print("best:", gs_gb.best_params_)
print("解读: test MAE 30.08 / R2 0.9722。对比基线 48.15：误差降 37.5%——"
      "综合题的『结果分』就压在这一行上")

assert gb_test[0] == 30.0752

## F.6 误差解剖：让数字说话

特征重要性：lag24 独占 **0.9152**，温度不足 1%——「明天≈今天」是负荷
预测的真相，做特征要有证据。分时段：午高峰 13 点误差 59.92 vs
夜间 22 点 15.43，峰谷差 3 倍——下一步该做峰值加权或分时段建模。

In [ ]:
# 特征重要性：lag24 独大 + 温度垫底的启示
rf100 = models["RF100"]
imp_rank = sorted(zip(FEATS, rf100.feature_importances_), key=lambda x: -x[1])
print([(k, round(float(v), 4)) for k, v in imp_rank])
print("解读: lag24 独占 91.5%——负荷预测的真相是『明天≈今天』；"
      "温度不足 1% 说明本数据温度-负荷相关弱。做特征要有证据，别凭直觉堆")

assert [k for k, _ in imp_rank[:3]] == ["lag24", "lag168", "dow"]

In [ ]:
# 误差解剖：分时段报告是加分项
te_an = te.copy()
te_an["pred"] = gs_gb.best_estimator_.predict(Xte)
te_an["err"] = (te_an["pred"] - te_an[TARGET]).abs()
err_by_hour = te_an.groupby("hour")["err"].mean()
print("最差小时:", int(err_by_hour.idxmax()), round(float(err_by_hour.max()), 2),
      "| 最好小时:", int(err_by_hour.idxmin()), round(float(err_by_hour.min()), 2))
print("解读: 上午峰 11 点误差 51.5 vs 深夜 23 点 13.12——尖峰追不上是负荷预测"
      "的通病，下一步该做峰值加权或分时段建模")

assert int(err_by_hour.idxmax()) == 11 and int(err_by_hour.idxmin()) == 23

## F.7 收官交付

冠军 GBR 落盘 `final_gbr.joblib`，读档指纹一致；最终成绩
**MAE 30.0752 / RMSE 39.8829 / R² 0.9722**。清洗→特征→切分→对比→
调参→解剖→落盘，七步闭环。

In [ ]:
# 收官：落盘 + 指纹 + 一图流
import joblib
best_model = gs_gb.best_estimator_
joblib.dump(best_model, "final_gbr.joblib")
loaded = joblib.load("final_gbr.joblib")
print("指纹一致:", pred_md5(best_model.predict(Xte)) == pred_md5(loaded.predict(Xte)))
print("最终成绩: MAE=30.0752 RMSE=39.8829 R2=0.9722")
print("解读: 综合题的完整闭环——清洗→特征→切分→对比→调参→解剖→落盘，"
      "七步每步都有 assert 兜底，这就是满分答卷的骨架")

assert pred_md5(best_model.predict(Xte)) == pred_md5(loaded.predict(Xte))

## 小结

1. 时序清洗三件套：datetime 化、排序、组内插值；先清洗后建模单独占分。
2. 特征工程的时序眼：lag24/lag168/roll24_mean；滚动窗口先 shift(1)。
3. 切分只有一种合法姿势：按时间，不许 shuffle；基线先行。
4. 树集成 > 线性；GBR 残差接力在平滑信号上最顺。
5. TimeSeriesSplit 的 cv 分数天然比 test 惨：早期折又旧又短。
6. 报告要分时段：峰谷误差差 3 倍，平均数会撒谎。
7. 交付闭环：joblib 落盘 + 指纹验收，只 load 不 fit。